# 🫀 Heart Disease Risk Analysis & Predictive Modeling (2026 Dataset)

**Data Science & Machine Learning Pipeline**
- **Dataset**: `heart_disease_risk_2026.csv` (9,000 patient records, 27 features)
- **Objective**: Predict `has_heart_disease` (0 = Low/No Risk, 1 = High Risk) using clinical, lab, lifestyle, and wearable metrics.
- **Pipeline**: Exploratory Data Analysis (EDA) ➔ Preprocessing & Encoding ➔ Multi-Model Benchmark (Logistic Regression, Random Forest, Gradient Boosting, XGBoost) ➔ Hyperparameter Tuning ➔ Model Serialization (`model.pkl`).

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, 
    confusion_matrix, classification_report, roc_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
print("✅ Data Science environment initialized successfully!")

--- 
## 1. Data Ingestion & Initial Inspection

In [ ]:
# Load dataset
df = pd.read_csv('heart_disease_risk_2026.csv')
print(f"Dataset Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")

print("\n--- First 5 Rows ---")
display(df.head())

print("\n--- Summary Info ---")
df.info()

print("\n--- Target Distribution (has_heart_disease) ---")
target_dist = df['has_heart_disease'].value_counts(normalize=True) * 100
print(target_dist)

--- 
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Missing values audit
missing = df.isnull().sum()
print("Missing Values Audit:")
print(missing[missing > 0] if missing.sum() > 0 else "Zero missing values found in the dataset!")

# Statistical summary
display(df.describe().T)

# Target class plot
plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df, x='has_heart_disease', palette=['#10b981', '#ef4444'])
plt.title('Distribution of Heart Disease Cases', fontsize=14, fontweight='bold')
plt.xlabel('Heart Disease (0 = No, 1 = Yes)')
plt.ylabel('Patient Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', fontsize=11, color='white', fontweight='bold')
plt.tight_layout()
plt.savefig('target_distribution.png')
plt.show()

--- 
## 3. Clinical & Lifestyle Feature Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=df, x='has_heart_disease', y='age', ax=axes[0, 0], palette=['#10b981', '#ef4444'])
axes[0, 0].set_title('Age vs Heart Disease Risk', fontweight='bold')

sns.boxplot(data=df, x='has_heart_disease', y='resting_bp_systolic', ax=axes[0, 1], palette=['#10b981', '#ef4444'])
axes[0, 1].set_title('Systolic BP vs Heart Disease Risk', fontweight='bold')

sns.boxplot(data=df, x='has_heart_disease', y='cholesterol_total', ax=axes[1, 0], palette=['#10b981', '#ef4444'])
axes[1, 0].set_title('Total Cholesterol vs Heart Disease Risk', fontweight='bold')

sns.boxplot(data=df, x='has_heart_disease', y='st_depression', ax=axes[1, 1], palette=['#10b981', '#ef4444'])
axes[1, 1].set_title('ST Depression vs Heart Disease Risk', fontweight='bold')

plt.tight_layout()
plt.savefig('feature_boxplots.png')
plt.show()

In [ ]:
# Chest Pain Type distribution
plt.figure(figsize=(9, 4.5))
sns.countplot(data=df, x='chest_pain_type', hue='has_heart_disease', palette=['#10b981', '#ef4444'])
plt.title('Chest Pain Type vs Heart Disease Risk', fontsize=14, fontweight='bold')
plt.xlabel('Chest Pain Category')
plt.ylabel('Patient Count')
plt.legend(title='Heart Disease', labels=['No Risk (0)', 'High Risk (1)'])
plt.tight_layout()
plt.savefig('chest_pain_analysis.png')
plt.show()

--- 
## 4. Data Preprocessing & Pipeline Construction

In [ ]:
# Define Features & Target
X = df.drop(columns=['patient_id', 'has_heart_disease'])
y = df['has_heart_disease']

num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'str']).columns.tolist()
bool_features = X.select_dtypes(include=['bool']).columns.tolist()

print(f"Numerical Features ({len(num_features)}): {num_features}")
print(f"Categorical Features ({len(cat_features)}): {cat_features}")
print(f"Boolean Features ({len(bool_features)}): {bool_features}")

# ColumnTransformer for Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
        ('bool', OneHotEncoder(handle_unknown='ignore', sparse_output=False), bool_features)
    ]
)

# Train / Test Split (80% Train, 20% Test, Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

--- 
## 5. Machine Learning Model Comparison & Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=42, eval_metric='logloss')
}

results = []
trained_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc
    })
    trained_pipelines[name] = pipeline

results_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False)
print("\n--- Machine Learning Benchmark Results ---")
display(results_df)

--- 
## 6. Best Model Selection & Detailed Diagnostics

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_pipeline = trained_pipelines[best_model_name]
print(f"🏆 Selected Best Performing Model: {best_model_name}")

y_pred_best = best_pipeline.predict(X_test)
y_proba_best = best_pipeline.predict_proba(X_test)[:, 1]

print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred_best, target_names=['Low Risk', 'High Risk']))

# Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6.5, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Low Risk', 'Predicted High Risk'],
            yticklabels=['Actual Low Risk', 'Actual High Risk'])
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

# ROC Curve Plot
plt.figure(figsize=(7, 5))
for name, pipe in trained_pipelines.items():
    prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    score = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {score:.4f})')

plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison Across Models', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png')
plt.show()

--- 
## 7. Model Serialization (`model.pkl` Export)

In [ ]:
# Export artifacts to model.pkl
model_export = {
    'pipeline': best_pipeline,
    'model_name': best_model_name,
    'feature_names': X.columns.tolist(),
    'num_features': num_features,
    'cat_features': cat_features,
    'bool_features': bool_features,
    'benchmark_metrics': results_df.to_dict(orient='records')
}

pickle_filename = 'model.pkl'
with open(pickle_filename, 'wb') as f:
    pickle.dump(model_export, f)

print(f"🎉 Successfully serialized model pipeline to '{pickle_filename}'!")

# Verification of serialized pickle file
with open(pickle_filename, 'rb') as f:
    loaded = pickle.load(f)

test_sample = X_test.iloc[:1]
pred = loaded['pipeline'].predict(test_sample)[0]
prob = loaded['pipeline'].predict_proba(test_sample)[0, 1]
print(f"✅ Pickle Load Verification Successful: Prediction={pred}, Risk Probability={prob:.2%}")